# 3.2 LangChain에서 도구(tool) 활용 방법

- LangChain에서 도구(tool)을 활용하는 방법을 알아봅니다
- 이 강의에서는 도구를 활용하는 방법을 중점적으로 다루며, 도구를 활용한 에이전트 개발 방법은 3.3 강의에서 다룹니다
    - LangGraph에서의 도구 활용 방법은 LangChain의 문법을 따르니 주의깊게 봐주세요

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model='gpt-4o',
    temperature=0
)

small_llm = ChatOpenAI(
    model='gpt-4o-mini',
    temperature=0
)

- [tool decorator](https://python.langchain.com/docs/how_to/custom_tools/#tool-decorator)를 사용하면 쉽게 도구를 만들 수 있습니다

In [ ]:
from langchain_core.tools import tool # tool도 invoke를 한다

@tool
def add(a: int, b: int) -> int:
    """숫자 a와 b를 더합니다."""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """숫자 a와 b를 곱합니다."""
    return a * b

- LLM을 호출했을 때와 도구를 사용했을 때의 차이를 알아봅니다

In [4]:
query = '3 곱하기 5는?'

In [ ]:
small_llm.invoke(query) # 도구 없이 content에 답을 해주는 과정

AIMessage(content='3 곱하기 5는 15입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 15, 'total_tokens': 26, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a1ddba3226', 'finish_reason': 'stop', 'logprobs': None}, id='run-fd4538b4-cd7c-4b88-a397-4260e0bc4608-0', usage_metadata={'input_tokens': 15, 'output_tokens': 11, 'total_tokens': 26, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

- 도구 리스트는 LLM에 해당하는 `BaseModel` 클래스에 `bind_tools` 메서드를 통해 전달합니다

In [6]:
llm_with_tools = small_llm.bind_tools([add, multiply]) # llm에 도구를 bind하고 리스트로 넘겨준다

- `AIMessage`의 `additional_kwargs` 속성은 `tool_calls`를 포함합니다
- `tool_calls`는 도구를 호출하는 메시지를 포함합니다

In [7]:
result = llm_with_tools.invoke(query) # llm이 도구를 호출하게 하는 방법
result # ai content는 비어있고, addtional_kwars의 tool_calls에 도구가 호출된 결과가 담긴다.

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_MQ4XM1qSEADRCvsK3SuD4V63', 'function': {'arguments': '{"a":3,"b":5}', 'name': 'multiply'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 84, 'total_tokens': 101, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_664ed7a1cd', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-022b2478-8087-4f69-9124-f9bcc0a555eb-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 5}, 'id': 'call_MQ4XM1qSEADRCvsK3SuD4V63', 'type': 'tool_call'}], usage_metadata={'input_tokens': 84, 'output_tokens': 17, 'total_tokens': 101, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

- 여기서 `tool_calls`의 형태를 기억해두시면 남은 강의를 이해하시는데 도움이 됩니다

In [ ]:
result.tool_calls # llm이 도구를 호출한 결과는 tool_calls에 담긴다. 도구가 여러개일 때는 리스트로 담긴다.

[{'name': 'multiply',
  'args': {'a': 3, 'b': 5},
  'id': 'call_MQ4XM1qSEADRCvsK3SuD4V63',
  'type': 'tool_call'}]

In [ ]:
from typing import Sequence

from langchain_core.messages import AnyMessage, HumanMessage

# message: Sequnce[AnyMessage] = [HumanMessage(query)] # message는 AnyMessage의 시퀀스 형태로 만들어준다.
# HumanMessage는 content를 담는 역할을 한다.

human_message = HumanMessage(query)
message_list: Sequence[AnyMessage] = [human_message]  # AnyMessage: AIMessage, HumanMessage, SystemMessage 등 모두 가능


- `tool_calls` 속성은 도구를 호출하는 메시지를 포함합니다
- `tool_calls`를 가진 `AIMessage`의 형태를 기억해두시면 남은 강의를 이해하시는데 도움이 됩니다

In [ ]:
ai_message = llm_with_tools.invoke(message_list) # human, ai, tool -> llm이 도구를 호출한 결과는 tool_calls에 담아야 함. 지금은 human만 담음
ai_message

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_51irH5qFtOvX7q7UGYC4BlSs', 'function': {'arguments': '{"a":3,"b":5}', 'name': 'multiply'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 84, 'total_tokens': 101, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_664ed7a1cd', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-2741e425-eb9e-4fcc-acfa-e83cd370a547-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 5}, 'id': 'call_51irH5qFtOvX7q7UGYC4BlSs', 'type': 'tool_call'}], usage_metadata={'input_tokens': 84, 'output_tokens': 17, 'total_tokens': 101, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [11]:
ai_message.tool_calls

[{'name': 'multiply',
  'args': {'a': 3, 'b': 5},
  'id': 'call_51irH5qFtOvX7q7UGYC4BlSs',
  'type': 'tool_call'}]

In [12]:
message_list.append(ai_message) # ai_message 추가

- `AIMessage`의 `tool_calls`를 활용해서 도구를 직접 호출할 수도 있습니다

In [13]:
tool_message = multiply.invoke(ai_message.tool_calls[0]) # 도구의 호출을 ai_message로 하는 것

In [ ]:
tool_message # tool_message도 message list에 append 해주어야 함

ToolMessage(content='15', name='multiply', tool_call_id='call_51irH5qFtOvX7q7UGYC4BlSs')

- 하지만 에이전트의 경우 도구를 직접 호출하는 것이 아니라 도구를 호출하는 메시지를 만들어서 전달합니다

In [15]:
message_list.append(tool_message) # tool_message 추가

In [16]:
llm_with_tools.invoke(message_list) # human, ai, tool -> llm이 도구를 호출한 결과는 tool_calls에 담아야 함. 지금은 human, ai, tool 모두 담음

AIMessage(content='3 곱하기 5는 15입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 109, 'total_tokens': 121, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_664ed7a1cd', 'finish_reason': 'stop', 'logprobs': None}, id='run-f9b11068-4ee4-4e7a-b0b7-b9164cd4e662-0', usage_metadata={'input_tokens': 109, 'output_tokens': 12, 'total_tokens': 121, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

- `message_list`의 순서를 기억해두시면 남은 강의를 이해하시는데 도움이 됩니다

In [17]:
message_list

[HumanMessage(content='3 곱하기 5는?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_51irH5qFtOvX7q7UGYC4BlSs', 'function': {'arguments': '{"a":3,"b":5}', 'name': 'multiply'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 84, 'total_tokens': 101, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_664ed7a1cd', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-2741e425-eb9e-4fcc-acfa-e83cd370a547-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 5}, 'id': 'call_51irH5qFtOvX7q7UGYC4BlSs', 'type': 'tool_call'}], usage_metadata={'input_tokens': 84, 'output_tokens': 17, 'total_tokens': 101, 'input_token_details': {'audio': 0